# Sentiment classification: TF-IDF vs. spaCy embeddings

## Setup

### Import TensorFlow and other libraries

In [3]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 60.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
import pandas as pd
import numpy as np
import re
import spacy

nlp = spacy.load('en_core_web_md')
tf.random.set_seed(42)

### Load the dataset

Change the following line to run this notebook on your own data.

In [5]:
from google.colab import files

uploaded = files.upload()
path_to_file = 'all_data.csv'
data = pd.read_csv(path_to_file)
data = data.dropna(subset=['review', 'sentiment']).reset_index(drop=True)
print(f"Number of reviews: {len(data)}")

Saving all_data.csv to all_data.csv
Number of reviews: 40516


### Look at the data

First, look at a few reviews:

In [6]:
for text in data['review'].head(5):
    print(repr(text))

'Aditya Ingole Deaf'
'I love the app.! There is no issue but if u could add the feature of do not spotlight on unmute in Android devices then it would be more nice'
'So hard to use. The web app failed, and the mobile app made me create an account before I could join the conference I was invited to. Poor experience all around. How do I delete this account?'
'I hate that the app makes a sound every time someone comes in the room. Sounds cut out if more than one person speaks at a time. I like Zoom better.'
'Useless at BSE star MF meet.voice too mych slow cant hear properly, and there was connection issues too.'


In [7]:
data['sentiment'].value_counts()

,count
sentiment,
2,15302
0,13466
1,11748


## Process the text

### Clean the text

Before building either representation, we clean the text the same way for both, so the comparison stays fair.

In [8]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data['clean_text'] = data['review'].map(preprocess_text)
data = data[data['clean_text'].str.len() > 0].reset_index(drop=True)
print(f"Reviews after cleaning: {len(data)}")

Reviews after cleaning: 39804


### Create training and test sets

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    data['clean_text'], data['sentiment'], test_size=0.2, random_state=42, stratify=data['sentiment']
)
y_train_arr = np.array(y_train)
y_test_arr = np.array(y_test)
print(f"Train examples: {len(X_train)}, Test examples: {len(X_test)}")

Train examples: 31843, Test examples: 7961


## Build the representations

### Representation 1: TF-IDF

A sparse, fixed-length representation built from word frequency across the corpus.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train).toarray().astype('float32')
X_test_tfidf = tfidf.transform(X_test).toarray().astype('float32')
print(f"TF-IDF shape: {X_train_tfidf.shape}")

TF-IDF shape: (31843, 5000)


### Representation 2: spaCy word embeddings

A dense, fixed-length representation: the average of pretrained word vectors for each review.

In [11]:
X_train_emb = np.array([doc.vector for doc in nlp.pipe(X_train, batch_size=256)], dtype='float32')
X_test_emb = np.array([doc.vector for doc in nlp.pipe(X_test, batch_size=256)], dtype='float32')
print(f"spaCy embeddings shape: {X_train_emb.shape}")

spaCy embeddings shape: (31843, 300)


## Build the model

Use `tf.keras.Sequential` to define the model, exactly the same architecture for both representations, so any performance difference reflects the representation, not the model.

In [12]:
def build_model(input_dim, num_classes=3):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Input(shape=(input_dim,)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [13]:
model_tfidf = build_model(input_dim=X_train_tfidf.shape[1])
model_tfidf.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       640,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 648,579 (2.47 MB)

 Trainable params: 648,579 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

## Train the model

In [14]:
history_tfidf = model_tfidf.fit(X_train_tfidf, y_train_arr, epochs=8, batch_size=128,
                                 validation_split=0.1, verbose=0)
print("Final training accuracy:", history_tfidf.history['accuracy'][-1])

Final training accuracy: 0.8646799921989441


In [15]:
model_emb = build_model(input_dim=X_train_emb.shape[1])
history_emb = model_emb.fit(X_train_emb, y_train_arr, epochs=8, batch_size=128,
                             validation_split=0.1, verbose=0)
print("Final training accuracy:", history_emb.history['accuracy'][-1])

Final training accuracy: 0.568253219127655


## Evaluate

In [16]:
from sklearn.metrics import f1_score, classification_report

test_loss_tfidf, test_acc_tfidf = model_tfidf.evaluate(X_test_tfidf, y_test_arr, verbose=0)
pred_tfidf = np.argmax(model_tfidf.predict(X_test_tfidf, verbose=0), axis=1)
f1_tfidf = f1_score(y_test_arr, pred_tfidf, average='macro')
print(f"TF-IDF -> accuracy: {test_acc_tfidf:.4f}, macro F1: {f1_tfidf:.4f}")
print(classification_report(y_test_arr, pred_tfidf))

TF-IDF -> accuracy: 0.6865, macro F1: 0.6726
              precision    recall  f1-score   support

           0       0.73      0.72      0.72      2657
           1       0.61      0.50      0.55      2302
           2       0.70      0.80      0.75      3002

    accuracy                           0.69      7961
   macro avg       0.68      0.67      0.67      7961
weighted avg       0.68      0.69      0.68      7961



In [17]:
test_loss_emb, test_acc_emb = model_emb.evaluate(X_test_emb, y_test_arr, verbose=0)
pred_emb = np.argmax(model_emb.predict(X_test_emb, verbose=0), axis=1)
f1_emb = f1_score(y_test_arr, pred_emb, average='macro')
print(f"spaCy embeddings -> accuracy: {test_acc_emb:.4f}, macro F1: {f1_emb:.4f}")
print(classification_report(y_test_arr, pred_emb))

spaCy embeddings -> accuracy: 0.5596, macro F1: 0.5181
              precision    recall  f1-score   support

           0       0.53      0.69      0.60      2657
           1       0.46      0.21      0.29      2302
           2       0.62      0.71      0.66      3002

    accuracy                           0.56      7961
   macro avg       0.54      0.54      0.52      7961
weighted avg       0.54      0.56      0.53      7961



## Try the classifiers on a new review

The following code block runs both classifiers on a review that isn't in the dataset:

In [18]:
def classify_review(text):
    clean = preprocess_text(text)
    mapping = {0: 'negative', 1: 'neutral', 2: 'positive'}
    tfidf_vec = tfidf.transform([clean]).toarray().astype('float32')
    emb_vec = np.array([nlp(clean).vector], dtype='float32')
    tfidf_pred = mapping[np.argmax(model_tfidf.predict(tfidf_vec, verbose=0), axis=1)[0]]
    emb_pred = mapping[np.argmax(model_emb.predict(emb_vec, verbose=0), axis=1)[0]]
    return {'tfidf_prediction': tfidf_pred, 'spacy_prediction': emb_pred}

classify_review("I really like the new interface, everything feels faster now")

{'tfidf_prediction': 'neutral', 'spacy_prediction': 'negative'}

In [19]:
classify_review("keeps freezing during calls, very frustrating")

{'tfidf_prediction': 'negative', 'spacy_prediction': 'negative'}

## Summary

The easiest way to improve either representation further would be to try more data cleaning steps (removing
stopwords, stemming/lemmatization), tuning `max_features` for TF-IDF, or trying a fine-tuned/domain-specific
embedding model instead of a general-purpose one. In this experiment, with identical preprocessing and the
identical `tf.keras` classifier, the results above show which representation held up better on this domain
of short, noisy, informal app reviews.